In [1]:
# we have exported the human-annotated data from the EPPI reviewer in the html format (configurable report tab)
import html
import csv
import json
import pandas as pd

In [2]:
# read_html creates a list of dataframes from each page but there is only 1 table in each, so we just choose the first
#df = pd.read_html("/home/lt1412/Downloads/sdg13_group1.html")[0]
files = ["/home/lt1412/Downloads/sdg13_group1.html", "/home/lt1412/Downloads/sdg13_group2.html", "/home/lt1412/Downloads/sdg13_group3.html", "/home/lt1412/Downloads/sdg13_group5.html"]
df = pd.concat([pd.read_html(f)[0] for f in files], ignore_index=True)

In [3]:
df = df.drop(columns=['Year','Abstract', 'Short Title'])
df = df.rename(columns={
    "Imported Id":"document_id", 
    "Title": "name"                    
                        })

display(df)
# TODO: check, somehow, there is less than 156 x 4 unique item IDs

,Item ID,document_id,name,SDG13,SDG13 - notes
0,117666036,458,2024 Annual Evaluation Synthesis Report,NaN,NaN
1,117667663,11012,A meta-analysis of evaluations managed by UN W...,NaN,NaN
2,117673147,14434,A STUDY OF THE AFGHANISTAN TRANSITION INITIATI...,NaN,NaN
3,117667636,10861,Action to Promote the Legal Empowerment of Wom...,NaN,NaN
4,117670544,13015,AmÃ©liorer les capacitÃ©s institutionnelles po...,NaN,NaN
...,...,...,...,...,...
617,117661970,590,UNESCO Synthetic Review of Evaluations 2021,NaN,NaN
618,117663020,13961,USAID OFFICE OF TRANSITION INITIATIVES SOMALIA...,NaN,NaN
619,117670486,12948,Women's Entrepreneurship Development and Gende...,NaN,NaN
620,117660971,981,Working Toward a Greener Global Recovery - Sev...,NaN,NaN


In [4]:
# process EPPI values into binary values
# if there is text - assign 1, if not - assign 0
df['SDG13'] = df['SDG13'].notna().astype(int)

In [5]:
df.value_counts('SDG13') # 36 out of 624ish 

SDG13
0    586
1     36
Name: count, dtype: int64

In [7]:
df.to_csv("/home/lt1412/Desktop/evaluation-reports-synthesis/data/labelled_data.csv", index=False)

In [2]:
df = pd.read_csv("/home/lt1412/Desktop/evaluation-reports-synthesis/data/labelled_data.csv")

In [ ]:
# create the first dev set
import random

# create an initial development set 
# keep sampling until condition is met
while True:
    subset = df.sample(n=5)
    if subset['SDG13'].eq(1).any():
        break

In [ ]:
subset.to_csv("data/development_labelled_data.csv")
# TODO: check they have PDFs, if needed re-sample

In [3]:
prev_samples = pd.read_csv("~/Desktop/evaluation-reports-synthesis/data/development_labelled_data.csv")

In [12]:
# create a development set, and drop / rename columns to match the model output

def create_dev_sample(df, prev_samples):
    while True:
        unseen_df  = df[~df['Item ID'].isin(prev_samples['Item ID'])]
        subset = unseen_df.sample(n=5)
        if subset['SDG13'].eq(1).any():
            subset = subset.rename(columns={'SDG13': 'sdg13'})
            subset = subset.drop(columns=['SDG13 - notes'])
            break

    return subset

In [14]:
subset = create_dev_sample(df, prev_samples)
subset.to_csv("/home/lt1412/Desktop/evaluation-reports-synthesis/data/2development_labelled_data.csv")